In [ ]:
!pip install --quiet vllm==0.11.0 transformers==4.45.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00
ERROR: Cannot install transformers==4.45.2 and vllm==0.11.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [ ]:
import torch
from vllm import LLM, SamplingParams

INFO 10-08 13:27:46 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
# 사용할 프롬프트 목록
prompts = [
    "대한민국의 수도는 어디인가요?",
    "LLM 서빙 최적화 기법에는 어떤 것들이 있나요?",
    "인공지능이 세상을 어떻게 바꿀까요? 한 문단으로 요약해줘.",
]

# 샘플링 파라미터 설정
# temperature: 높을수록 창의적이고 무작위적인 텍스트 생성
# top_p: 확률 분포의 누적값이 p가 될 때까지의 토큰만 고려하여 샘플링
# max_tokens: 생성할 최대 토큰 수
sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=256)

# dtype="auto"는 사용 가능한 GPU에 맞춰 자동으로 정밀도를 설정합니다. (예: float16)
llm = LLM(model="Qwen/Qwen1.5-1.8B-Chat", dtype="float16",  max_model_len=1024,   gpu_memory_utilization=0.95)
print("모델 로드가 완료되었습니다.")

WARNING 10-08 13:36:34 [config.py:2836] Casting torch.bfloat16 to torch.float16.
INFO 10-08 13:36:34 [config.py:689] This model supports multiple tasks: {'embed', 'score', 'generate', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 10-08 13:36:34 [llm_engine.py:243] Initializing a V0 LLM engine (v0.8.4) with config: model='Qwen/Qwen1.5-1.8B-Chat', speculative_config=None, tokenizer='Qwen/Qwen1.5-1.8B-Chat', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metrics=False, otlp_traces_endpoint=None, collec

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

INFO 10-08 13:39:37 [weight_utils.py:281] Time spent downloading weights for Qwen/Qwen1.5-1.8B-Chat: 180.160335 seconds
INFO 10-08 13:39:37 [weight_utils.py:315] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 10-08 13:39:54 [loader.py:458] Loading weights took 16.96 seconds
INFO 10-08 13:39:54 [model_runner.py:1146] Model loading took 3.4654 GiB and 197.928260 seconds
INFO 10-08 13:39:57 [worker.py:267] Memory profiling takes 1.81 seconds
INFO 10-08 13:39:57 [worker.py:267] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.95) = 14.00GiB
INFO 10-08 13:39:57 [worker.py:267] model weights take 3.47GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 1.39GiB; the rest of the memory reserved for KV Cache is 9.12GiB.
INFO 10-08 13:39:57 [executor_base.py:112] # cuda blocks: 3112, # CPU blocks: 1365
INFO 10-08 13:39:57 [executor_base.py:117] Maximum concurrency for 1024 tokens per request: 48.62x
INFO 10-08 13:40:02 [model_runner.py:1456] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 10-08 13:40:38 [model_runner.py:1598] Graph capturing finished in 36 secs, took 0.16 GiB
INFO 10-08 13:40:38 [llm_engine.py:449] init engine (profile, create kv cache, warmup model) took 44.04 seconds
모델 로드가 완료되었습니다.


In [ ]:
# 모델의 토크나이저를 가져옵니다.
tokenizer = llm.get_tokenizer()

# 각 프롬프트를 Qwen 모델의 공식 채팅 형식으로 변환합니다.
# [{"role": "user", "content": prompt}] 형식으로 대화 턴을 구성합니다.
# add_generation_prompt=True는 마지막에 모델이 답변을 생성할 차례임을 알려줍니다.
formatted_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True
    ) for p in prompts
]

In [ ]:
print("Chat Template을 적용하여 추론을 시작합니다...")
outputs = llm.generate(formatted_prompts, sampling_params)
print("추론이 완료되었습니다.\n")

Chat Template을 적용하여 추론을 시작합니다...


Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

추론이 완료되었습니다.



In [ ]:
for i, (prompt, output) in enumerate(zip(prompts, outputs)):
    generated_text = output.outputs[0].text.strip()
    print("-" * 60)
    # enumerate를 사용해 질문 번호를 붙입니다.
    print(f"✅ [질문 {i+1}]")
    print(prompt)
    print("\n✅ [답변]")
    print(generated_text)
    print("-" * 60, "\n")

------------------------------------------------------------
✅ [질문 1]
대한민국의 수도는 어디인가요?

✅ [답변]
대한민국의 수도는 서울입니다. 서울은 한국의 수도 중 가장 큰 도시이며, 개발된 역사와 문화로 인해 유명한 도시 중 하나입니다. 서울은 한국의 수도 중 가장 큰 도시이며, 수도권은 아시아 최대 규모의 도시 중 하나로, 서울의 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울시의 수도권으로, 수도권은 서울
------------------------------------------------------------ 

------------------------------------------------------------
✅ [질문 2]
LLM 서빙 최적화 기법에는 어떤 것들이 있나요?

✅ [답변]
LLM 서빙 최적화는 학업 프로그램의 성공적인 운영을 위해 학생들이 학습하고 배우는 과정에서 다양한 요소를 고려하는 것을 포함한 고려된 학습 방법을 개발하는 것을 의미합니다. 이러한 요소는 다음과 같습니다.

1. 학습 목표 설정: 학생들이 학습 목표를 설정하고 달성할 것입니다. 이는 학생들이 학습하는 과정에서 무엇을 달성할지, 학생들이 필요한 지식과 기술을 배우는지, 학생들이 자신의 학습 계획을 세우는지 등에 대한 목표를 설정합니다.

2. 학습 자료 선택: 학생들이 학습할 과정에서 필요한 학습 자료를 선택하고 소유하고 사